# 🚗 Semantic Segmentation on IDD Dataset
## DeepLabV3+ with ResNet-50 Encoder

---

This notebook implements a **semantic segmentation** pipeline for the **Indian Driving Dataset (IDD)** using the `DeepLabV3+` architecture.

| Item | Detail |
|------|--------|
| **Task** | Semantic Segmentation |
| **Dataset** | IDD (Indian Driving Dataset) |
| **Model** | DeepLabV3+ |
| **Encoder** | ResNet-50 (SSL pretrained) |
| **Classes** | 10 (reduced from 30+) |
| **Input Size** | 512 × 512 |
| **Loss** | CrossEntropy + Dice (50/50) |

### 📋 Notebook Sections
1. Imports & Libraries
2. Class Mapping & Dataset Class
3. Image Loading Utility
4. Visualization Helper
5. Dataset Exploration
6. Augmentations & DataLoaders
7. Model Setup
8. Loss Functions
9. Optimizer & Scheduler
10. mIoU Metric
11. Training Loop
12. 📉 Loss & mIoU Curves
13. 📊 Per-Class IoU Bar Chart
14. 🎨 Prediction Visualization
15. 🎨 Class Color Legend

---
## 1. 📦 Imports & Libraries

Core libraries used throughout the project:
- **`torch`** — deep learning framework
- **`cv2`** — image I/O and color conversion
- **`albumentations`** — fast augmentation pipeline compatible with segmentation masks
- **`numpy`, `os`, `glob`** — general utilities

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from glob import glob
import numpy as np
import os

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

---
## 2. 🗺️ Class Mapping & Dataset Definition

### Why reduce to 10 classes?
The original IDD has 30+ fine-grained labels. With limited GPU memory and training time, we merge them into **10 meaningful classes**:

| ID | Class | Note |
|----|-------|------|
| 0 | Road | Most common class |
| 1 | Sidewalk | |
| 2 | Person | Safety-critical |
| 3 | Two-Wheeler | Bikes, motorcycles |
| 4 | Large Vehicle | Cars, buses, trucks |
| 5 | Animal | Rare but important |
| 6 | Traffic Sign | |
| 7 | Building | |
| 8 | Sky | |
| 9 | Background | Everything else |

### IDDDataset
Custom `torch.utils.data.Dataset` that:
- Reads images as **RGB**
- Reads masks as **grayscale**
- Clips any pixel `>= 10` → class `9` (Background) to prevent CUDA out-of-range errors
- Applies augmentation transforms when provided

In [ ]:
NUM_Classes = 10

CLASS_MAPPING = {
    0: 0,   # drivable area / road
    1: 1,   # sidewalk
    2: 2,   # person / pedestrian
    3: 3,   # two-wheeler (bicycle, motorcycle, scooter)
    4: 4,   # large vehicle (car, bus, truck)
    5: 5,   # animal
    6: 6,   # traffic sign
    7: 7,   # building
    8: 8,   # sky
    9: 9,   # background / other
}

CLASS_NAMES = [
    "Road", "Sidewalk", "Person", "Two-Wheeler",
    "Large Vehicle", "Animal", "Traffic Sign",
    "Building", "Sky", "Background"
]


class IDDDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir    = image_dir
        self.mask_dir     = mask_dir
        self.transform    = transform
        self.images       = sorted(os.listdir(image_dir))
        self.masks_files  = sorted(os.listdir(mask_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path  = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir,  self.masks_files[idx])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Safety clip: pixel values >= 10 → class 9 (Background)
        # Prevents CUDA index out-of-range errors during training
        mask[mask >= 10] = 9

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask  = augmented['mask']

        return image, mask.long()

print(f"✅ IDDDataset defined  |  {NUM_Classes} classes")

---
## 3. 🖼️ Image Loading Utility

`load_image` is a simple helper used only for **exploratory visualization** — it is **not** part of the training pipeline.

- **Images**: read with `cv2` in BGR, then flipped to RGB
- **Masks**: read as single-channel grayscale

In [ ]:
def load_image(path, mask=False):
    """Load an image (RGB) or mask (grayscale) from disk for visualization."""
    img = cv2.imread(path, -1)
    if not mask:
        img = img[..., ::-1]          # BGR → RGB
    else:
        if len(img.shape) > 2:
            img = img[..., 0:1]       # keep single channel
    return img

print("✅ load_image() defined")

---
## 4. 📊 Visualization Helper — `show_sample`

Displays three panels per sample:
1. **Original Image**
2. **Segmentation Mask** (class IDs as a colormap)
3. **Overlay** — mask blended onto the image at 60% opacity

In [ ]:
import matplotlib.pyplot as plt

def show_sample(img, msk):
    """Render original image, mask, and blended overlay side-by-side."""
    msk = msk.squeeze()   # ensure 2-D

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title("Original Image", fontsize=13)
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(msk, cmap='tab10', vmin=0, vmax=9)
    plt.colorbar(label='Class ID', shrink=0.8)
    plt.title("Segmentation Mask", fontsize=13)
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(img)
    plt.imshow(np.ma.masked_where(msk == 0, msk),
               alpha=0.60, cmap='coolwarm_r', vmin=0, vmax=9)
    plt.title("Overlay (Image + Mask)", fontsize=13)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

print("✅ show_sample() defined")

---
## 5. 🔍 Dataset Exploration

We sample images at fixed intervals across the training set to visually verify:
- Images and masks are **correctly paired**
- Mask pixel values are **within the expected 0–9 range**
- Augmentations haven't corrupted anything

> ⚠️ **Note:** update the paths below to match your environment (e.g. Google Drive or Kaggle).

In [ ]:
import pandas as pd

# ── Update these paths to match your environment ──────────────────────────────
TRAIN_IMG_DIR  = 'Data/dataset/IDD/train/images'
TRAIN_MASK_DIR = 'Data/dataset/IDD/train/labels'
# ─────────────────────────────────────────────────────────────────────────────

image_paths = sorted(glob(f'{TRAIN_IMG_DIR}/*'))
label_paths = sorted(glob(f'{TRAIN_MASK_DIR}/*'))

df = pd.DataFrame(zip(image_paths, label_paths),
                  columns=['image_path', 'mask_path'])

print(f"Total training samples found: {len(df)}")

if len(df) == 0:
    print("⚠️  No images found — please check your data paths above.")
else:
    sample_indices = [i for i in range(0, min(200, len(df)), 25)]
    for i in sample_indices:
        row = df.iloc[i]
        img = load_image(row.image_path)
        msk = load_image(row.mask_path, mask=True)
        print(f"Sample {i}  |  image: {img.shape}  |  mask: {msk.shape}")
        show_sample(img, msk)

---
## 6. 🔄 Augmentations & DataLoaders

### Augmentation Strategy

| Transform | Train | Val | Purpose |
|-----------|:-----:|:---:|---------|
| `Resize(512,512)` | ✅ | ✅ | Uniform input size |
| `HorizontalFlip(p=0.5)` | ✅ | ❌ | Mirror invariance |
| `RandomScale` | ✅ | ❌ | Scale robustness |
| `PadIfNeeded + RandomCrop` | ✅ | ❌ | Crop diversity |
| `ColorJitter` | ✅ | ❌ | Lighting variation |
| `Normalize (ImageNet)` | ✅ | ✅ | Match encoder pretraining stats |

### DataLoader Settings
- `batch_size = 8` — increase to 16/32 if your GPU allows
- `shuffle = True` — mandatory during training to avoid ordering bias
- `pin_memory = True` — speeds up CPU → GPU transfer

In [ ]:
MEAN = [0.485, 0.456, 0.406]   # ImageNet mean
STD  = [0.229, 0.224, 0.225]   # ImageNet std

# --- Training augmentation (heavy) ---
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.RandomScale(scale_limit=(-0.5, 1.0), p=0.5),
    A.PadIfNeeded(min_height=512, min_width=512, border_mode=0, p=1.0),
    A.RandomCrop(512, 512),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.5),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

# --- Validation transform (resize + normalize only) ---
val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

# --- Datasets ---
trainDataset = IDDDataset(
    image_dir='Data/dataset/IDD/train/images',
    mask_dir ='Data/dataset/IDD/train/labels',
    transform=train_transform
)
valDataset = IDDDataset(
    image_dir='Data/dataset/IDD/valid/images',
    mask_dir ='Data/dataset/IDD/valid/labels',
    transform=val_transform
)

# --- DataLoaders ---
train_loader = DataLoader(
    trainDataset,
    batch_size=8,
    shuffle=True,       # randomise order each epoch
    num_workers=0,      # set to 4+ on multi-core machines
    pin_memory=True     # faster CPU→GPU transfer
)
val_loader = DataLoader(
    valDataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# --- Sanity check ---
images, masks = next(iter(train_loader))
print(f"Train size : {len(trainDataset)}")
print(f"Val size   : {len(valDataset)}")
print(f"Image batch: {images.shape}")   # [8, 3, 512, 512]
print(f"Mask batch : {masks.shape}")    # [8, 512, 512]

---
## 7. 🧠 Model — DeepLabV3+

### Architecture
- **Backbone / Encoder**: ResNet-50 pretrained with **SSL** (Self-Supervised Learning)
- **Decoder**: Atrous Spatial Pyramid Pooling (ASPP) + refinement head
- **Output**: 10-channel logit map → `argmax` → per-pixel class label

### Why SSL weights?
SSL pretraining on unlabelled data produces richer low-level features compared to plain ImageNet supervision, which is especially useful for domain-shifted datasets like Indian driving scenes.

We resume from a **saved checkpoint at epoch 10** to continue fine-tuning.

In [ ]:
import torch
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Build model ---
model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="ssl",   # Self-supervised pretrained weights
    in_channels=3,
    classes=NUM_Classes
)

# --- Load checkpoint ---
weights_path = "model_v2_epoch_10.pth"
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
print("✅ Pretrained weights loaded successfully!")

model = model.to(device)
model.eval()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model ready on {device}")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}")

---
## 8. ⚖️ Loss Functions — Combined CE + Dice

### The Class Imbalance Problem
In driving scenes, **road** and **sky** dominate pixel counts while **animals** and **pedestrians** are rare. An unweighted loss makes the model ignore rare but safety-critical classes.

### Solution: Weighted CrossEntropy + Dice

| Loss | Focus | Formula |
|------|-------|---------|
| CrossEntropy | Per-pixel accuracy | $-\sum w_c \cdot y_c \log \hat{y}_c$ |
| Dice | Region overlap | $1 - \frac{2|P \cap G|}{|P|+|G|}$ |

$$\mathcal{L} = 0.5 \cdot \mathcal{L}_{CE} + 0.5 \cdot \mathcal{L}_{Dice}$$

### Class Weights
Higher weight = more penalty when the model gets that class wrong.

| Class | Weight | Reason |
|-------|--------|--------|
| Road, Sky | 0.5 | Very common — don't over-penalize |
| Animal | 4.0 | Very rare — force the model to pay attention |
| Person, Traffic Sign | 3.0 | Safety-critical |

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# Per-class weights — inverse frequency heuristic
class_weights = torch.tensor([
    0.5,   # 0 - road           (very common)
    1.0,   # 1 - sidewalk
    3.0,   # 2 - pedestrian     (rare, safety-critical)
    2.5,   # 3 - two-wheeler
    1.5,   # 4 - large vehicle
    4.0,   # 5 - animal         (very rare)
    3.0,   # 6 - traffic sign
    0.8,   # 7 - building
    0.5,   # 8 - sky            (very common)
    1.0,   # 9 - background
]).to(device)

criterion_ce   = nn.CrossEntropyLoss(weight=class_weights)
criterion_dice = smp.losses.DiceLoss(mode='multiclass')

def combined_loss(pred, target):
    """50% weighted CrossEntropy + 50% Dice loss."""
    loss_ce   = criterion_ce(pred, target)    # classification accuracy
    loss_dice = criterion_dice(pred, target)  # region overlap quality
    return 0.5 * loss_ce + 0.5 * loss_dice

print("✅ Loss functions defined")
print("   CrossEntropyLoss : weighted per class frequency")
print("   DiceLoss         : multiclass mode")
print("   Combined         : 0.5 × CE  +  0.5 × Dice")

---
## 9. ⚙️ Optimizer & LR Scheduler

### AdamW
- `lr = 1e-4` — conservative to preserve SSL pretrained features
- `weight_decay = 1e-4` — L2 regularization

### Polynomial LR Decay
$$lr_t = lr_0 \times \left(1 - \frac{t}{T}\right)^{0.9}$$

Gradually reduces the learning rate over 50 epochs, which is standard practice in DeepLab-family models.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.PolynomialLR(
    optimizer,
    total_iters=50,
    power=0.9
)

print("✅ Optimizer : AdamW")
print(f"   lr           = {optimizer.param_groups[0]['lr']}")
print(f"   weight_decay = {optimizer.param_groups[0]['weight_decay']}")
print("✅ Scheduler : PolynomialLR  (total_iters=50, power=0.9)")

---
## 10. 📐 Evaluation Metric — Mean IoU

**IoU per class:**

$$IoU_c = \frac{|Pred_c \cap GT_c|}{|Pred_c \cup GT_c|}$$

**Mean IoU** averages across all *present* classes (classes absent in a batch are skipped to avoid division by zero):

$$mIoU = \frac{1}{C}\sum_{c=1}^{C} IoU_c$$

A higher mIoU means the predicted segments align better with ground truth.

In [ ]:
def compute_miou(preds, targets, num_classes=NUM_Classes):
    """
    Compute mean Intersection over Union (mIoU).

    Args:
        preds   : raw logits  (B, C, H, W)
        targets : class IDs   (B, H, W)
    Returns:
        float — mean IoU across all present classes
    """
    preds = torch.argmax(preds, dim=1)   # (B, H, W)
    iou_per_class = []

    for cls in range(num_classes):
        pred_cls   = (preds   == cls)
        target_cls = (targets == cls)
        intersection = (pred_cls & target_cls).sum().float()
        union        = (pred_cls | target_cls).sum().float()

        if union == 0:
            continue   # skip absent classes
        iou_per_class.append((intersection / union).item())

    return sum(iou_per_class) / len(iou_per_class) if iou_per_class else 0.0

print("✅ compute_miou() defined")

---
## 11. 🏋️ Training Loop

### Features
- **Mixed Precision (AMP)** — `autocast` + `GradScaler` gives ~2× speedup on modern GPUs
- **Validation phase** every epoch to monitor generalization
- **Loss & mIoU history** saved for plotting after training
- **Checkpoint saved** every epoch as `model_v2_epoch_N.pth`
- **Automatic CPU fallback** if GPU is unavailable

### Per-Epoch Flow
```
Train:
  for each batch → forward (FP16) → combined loss → backward → optimizer step

Validate:
  for each batch → forward (no grad) → loss + mIoU → accumulate

→ scheduler step → save checkpoint
```

In [ ]:
from tqdm import tqdm

# --- GPU check ---
try:
    torch.zeros(1).to(device)
    use_amp = True
    print("🚀 GPU ready — AMP enabled")
except Exception:
    print("⚠️  GPU unavailable — falling back to CPU")
    device = torch.device("cpu")
    model.to(device)
    use_amp = False

scaler = torch.amp.GradScaler('cuda')

# History lists — used for plotting after training
train_losses = []
val_losses   = []
val_mious    = []

epochs = 30

for epoch in range(epochs):

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch [{epoch+1:02d}/{epochs}]")

    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = model(images)
            loss    = combined_loss(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    v_loss = 0.0
    v_miou = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(images)
                loss    = combined_loss(outputs, masks)
            v_loss += loss.item()
            v_miou += compute_miou(outputs, masks)

    avg_val_loss = v_loss / len(val_loader)
    avg_val_miou = v_miou / len(val_loader)
    val_losses.append(avg_val_loss)
    val_mious.append(avg_val_miou)

    scheduler.step()

    print(f"  → Train Loss: {avg_train_loss:.4f}  |  "
          f"Val Loss: {avg_val_loss:.4f}  |  Val mIoU: {avg_val_miou:.4f}")

    torch.save(model.state_dict(), f"model_v2_epoch_{epoch+1}.pth")

print("\n✅ Training complete!")

---
## 12. 📉 Loss & mIoU Curves Over Time

These plots let us diagnose:
- **Overfitting** — val loss rises while train loss keeps falling
- **Convergence** — both curves plateau
- **Best epoch** — highlighted on the mIoU chart for checkpoint selection

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Training Progress", fontsize=16, fontweight='bold')

ep = range(1, len(train_losses) + 1)

# --- Loss curve ---
ax1 = axes[0]
ax1.plot(ep, train_losses, 'o-', color='#e74c3c', lw=2, ms=4, label='Train Loss')
ax1.plot(ep, val_losses,   's--', color='#3498db', lw=2, ms=4, label='Val Loss')
ax1.fill_between(ep, train_losses, val_losses, alpha=0.07, color='gray')
ax1.set_title("Loss Over Epochs", fontsize=13)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# --- mIoU curve ---
ax2 = axes[1]
ax2.plot(ep, val_mious, 'D-', color='#2ecc71', lw=2, ms=4, label='Val mIoU')
best_ep  = int(np.argmax(val_mious)) + 1
best_val = max(val_mious)
ax2.axvline(best_ep, color='orange', ls='--', lw=1.5,
            label=f'Best epoch ({best_ep})')
ax2.annotate(f"Best: {best_val:.4f}",
             xy=(best_ep, best_val),
             xytext=(best_ep + 0.6, best_val - 0.015),
             arrowprops=dict(arrowstyle='->', color='orange'),
             fontsize=10, color='darkorange')
ax2.set_title("Validation mIoU Over Epochs", fontsize=13)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("mIoU")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Best Val mIoU : {best_val:.4f}  at epoch {best_ep}")

---
## 13. 📊 Per-Class IoU Bar Chart

After training we run a full validation pass and compute **IoU per class** individually.  
This reveals which classes the model handles well and which ones still need improvement (usually rare classes like Animal or Traffic Sign).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model.eval()
cls_iou   = np.zeros(NUM_Classes)
cls_count = np.zeros(NUM_Classes)

with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        preds   = torch.argmax(outputs, dim=1)

        for cls in range(NUM_Classes):
            p = (preds  == cls)
            t = (masks  == cls)
            inter = (p & t).sum().float().item()
            union = (p | t).sum().float().item()
            if union > 0:
                cls_iou[cls]   += inter / union
                cls_count[cls] += 1

avg_cls_iou = np.where(cls_count > 0, cls_iou / cls_count, 0)

# --- Bar chart ---
fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.RdYlGn(avg_cls_iou)
bars = ax.bar(CLASS_NAMES, avg_cls_iou, color=colors,
              edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, avg_cls_iou):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{val:.3f}",
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(avg_cls_iou.mean(), color='steelblue', ls='--', lw=1.5,
           label=f'Mean IoU = {avg_cls_iou.mean():.3f}')
ax.set_title("Per-Class IoU — Validation Set", fontsize=14, fontweight='bold')
ax.set_xlabel("Class")
ax.set_ylabel("IoU Score")
ax.set_ylim(0, 1.1)
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("per_class_iou.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nPer-Class IoU Summary:")
for name, iou in zip(CLASS_NAMES, avg_cls_iou):
    filled = '█' * int(iou * 20)
    print(f"  {name:<16}: {iou:.4f}  {filled}")
print(f"\n  Overall mIoU : {avg_cls_iou.mean():.4f}")

---
## 14. 🎨 Visual Results — Predictions vs Ground Truth

Each row shows:
1. **Original Image** — the raw input frame
2. **Ground Truth** — human-annotated mask
3. **Model Prediction** — model output

Colors follow the IDD palette defined below. The per-sample mIoU score is shown on the left.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Color palette — one RGB color per class
CLASS_COLORS = np.array([
    [128,  64, 128],   # 0 - Road
    [244,  35, 232],   # 1 - Sidewalk
    [220,  20,  60],   # 2 - Person
    [255, 140,   0],   # 3 - Two-Wheeler
    [  0,   0, 142],   # 4 - Large Vehicle
    [  0, 200,   0],   # 5 - Animal
    [220, 220,   0],   # 6 - Traffic Sign
    [ 70,  70,  70],   # 7 - Building
    [ 70, 130, 180],   # 8 - Sky
    [ 81,   0,  81],   # 9 - Background
], dtype=np.uint8)

def decode_mask(mask):
    """Convert (H, W) class-ID mask → (H, W, 3) RGB image using CLASS_COLORS."""
    colour_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_id, colour in enumerate(CLASS_COLORS):
        colour_mask[mask == cls_id] = colour
    return colour_mask

def visualize_predictions(model, dataset, device, num_images=4, seed=42):
    """Plot original / ground-truth / prediction for random validation samples."""
    np.random.seed(seed)
    model.eval()

    fig, axes = plt.subplots(num_images, 3, figsize=(18, num_images * 5))
    fig.suptitle("Original  |  Ground Truth  |  Model Prediction",
                 fontsize=15, fontweight='bold')

    for col, title in enumerate(["Original Image", "Ground Truth", "Model Prediction"]):
        axes[0, col].set_title(title, fontsize=13, fontweight='bold', pad=8)

    for i in range(num_images):
        idx = np.random.randint(0, len(dataset))
        image, mask = dataset[idx]
        input_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():
            output     = model(input_tensor)
            prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

        # Denormalise for display
        img_np = image.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
        img_np = np.clip(img_np, 0, 1)

        # Compute per-sample mIoU
        sample_miou = compute_miou(output, mask.unsqueeze(0).to(device))

        axes[i, 0].imshow(img_np)
        axes[i, 1].imshow(decode_mask(mask.cpu().numpy()))
        axes[i, 2].imshow(decode_mask(prediction))

        for col in range(3):
            axes[i, col].axis('off')

        axes[i, 0].set_ylabel(f"Sample {idx}\nmIoU = {sample_miou:.3f}",
                              fontsize=10, rotation=0, labelpad=90, va='center')

    # Color legend at bottom
    patches = [mpatches.Patch(color=np.array(CLASS_COLORS[c]) / 255,
                               label=CLASS_NAMES[c]) for c in range(NUM_Classes)]
    fig.legend(handles=patches, loc='lower center', ncol=5,
               fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.04))

    plt.tight_layout()
    plt.savefig("prediction_results.png", dpi=150, bbox_inches='tight')
    plt.show()

visualize_predictions(model, valDataset, device, num_images=4)

---
## 15. 🎨 Class Color Legend

A standalone color swatch mapping each class ID to its display color.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.axis('off')
ax.set_title("Segmentation Class Color Legend", fontsize=14,
             fontweight='bold', pad=10)

for i, (name, colour) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    col = i % 5
    row = i // 5
    x = col * 0.20 + 0.01
    y = 0.80 - row * 0.45

    rect = plt.Rectangle((x, y), 0.04, 0.28,
                          color=np.array(colour) / 255,
                          transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)
    ax.text(x + 0.05, y + 0.13, f"{i}: {name}",
            transform=ax.transAxes, fontsize=10,
            va='center', ha='left')

plt.tight_layout()
plt.savefig("colour_legend.png", dpi=150, bbox_inches='tight')
plt.show()